# 01 · Inventory checks

Goal: open **one** precipitation file by hand and work out the units of `PREC`.

The full inventory of all 12 files comes from `python -m src.inventory` → `outputs/tables/file_inventory.csv`.

In [1]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.config import DATA_RAW, OUT_TABLES

pd.set_option("display.width", 200)

## 1. Inventory summary

In [2]:
inv = pd.read_csv(OUT_TABLES / "file_inventory.csv")
inv[["model", "scenario", "variable", "start_year", "end_year", "n_rows", "n_dates", "n_cells",
     "first_date", "last_date", "months_ok", "complete"]]

,model,scenario,variable,start_year,end_year,n_rows,n_dates,n_cells,first_date,last_date,months_ok,complete
0,BESM,historical,PREC,1961,2005,476280,540,882,1961-01-01,2005-12-01,True,True
1,BESM,RCP45,PREC,2006,2099,994896,1128,882,2006-01-01,2099-12-01,True,True
2,BESM,RCP85,PREC,2006,2099,994896,1128,882,2006-01-01,2099-12-01,True,True
3,CanESM2,historical,PREC,1961,2005,476280,540,882,1961-01-01,2005-12-01,True,True
4,CanESM2,RCP45,PREC,2006,2099,994896,1128,882,2006-01-01,2099-12-01,True,True
5,CanESM2,RCP85,PREC,2006,2099,994896,1128,882,2006-01-01,2099-12-01,True,True
6,HadGEM2-ES,historical,PREC,1961,2005,476280,540,882,1961-01-01,2005-12-01,True,True
7,HadGEM2-ES,RCP45,PREC,2006,2099,994014,1127,882,2006-01-01,2099-11-01,False,True
8,HadGEM2-ES,RCP85,PREC,2006,2099,994896,1128,882,2006-01-01,2099-12-01,True,True
9,MIROC5,historical,PREC,1961,2005,476280,540,882,1961-01-01,2005-12-01,True,True


## 2. Inspect one file by hand

`BESM_HISTORICO` covers the 1961–2005 baseline, so it can be compared with observed climatology.

In [3]:
f = DATA_RAW / "BESM_HISTORICO_PREC_1961_2005_PY.csv"
df = pd.read_csv(f)
df["date"] = pd.to_datetime(df["Data"], format="%Y/%m/%d")
print(df.shape)
df.head()

(476280, 6)


,Data,Hora,Latitude,Longitude,PREC,date
0,1961/01/01,00:00:00,-22.2,-56.4,88.180733,1961-01-01
1,1961/01/01,00:00:00,-22.2,-57.6,73.854874,1961-01-01
2,1961/01/01,00:00:00,-22.2,-57.8,65.743095,1961-01-01
3,1961/01/01,00:00:00,-22.4,-56.2,70.972977,1961-01-01
4,1961/01/01,00:00:00,-22.4,-56.4,79.482231,1961-01-01


In [4]:
df["PREC"].describe()

count    476280.000000
mean         65.724024
std          48.931822
min           0.000000
25%          29.838246
50%          55.739702
75%          90.388411
max         472.142090
Name: PREC, dtype: float64

In [5]:
print("min :", df["PREC"].min())
print("max :", df["PREC"].max())
print("mean:", round(df["PREC"].mean(), 2))
print("dates per year:", df.groupby(df["date"].dt.year)["date"].nunique().unique())
print("days of month present:", sorted(df["date"].dt.day.unique()))

min : 0.0
max : 472.14208984375
mean: 65.72
dates per year: [12]
days of month present: [np.int32(1)]


## 3. Infer units

There is one row per cell per month (always dated the 1st), so `PREC` is either a **monthly total (mm/month)** or a **monthly mean rate (mm/day)**.
Multiply the mean by 12 (if mm/month) or by 365 (if mm/day) and compare with Paraguay's known annual rainfall: about 400–900 mm/yr in the western Chaco and about 1,500–1,900 mm/yr in the east.

In [6]:
m = df["PREC"].mean()
print(f"annual if mm/month: {m * 12:8.0f} mm/yr")
print(f"annual if mm/day  : {m * 365:8.0f} mm/yr")

annual if mm/month:      789 mm/yr
annual if mm/day  :    23989 mm/yr


Annual totals by cell show the spatial gradient. With mm/month this sum is the annual rainfall in mm.

In [7]:
annual = (df.assign(year=df["date"].dt.year)
            .groupby(["Latitude", "Longitude", "year"])["PREC"].sum()
            .groupby(["Latitude", "Longitude"]).mean()
            .rename("annual_mm").reset_index())
annual["annual_mm"].describe()

count     882.000000
mean      788.688293
std       292.035145
min       478.641302
25%       552.739509
50%       651.834859
75%      1003.331503
max      1493.097093
Name: annual_mm, dtype: float64

In [8]:
# West (Chaco) vs East (Oriental): split at roughly the Paraguay River, about -58° longitude
annual["region"] = annual["Longitude"].lt(-58).map({True: "west (< -58°)", False: "east (>= -58°)"})
annual.groupby("region")["annual_mm"].describe()[["count", "mean", "min", "max"]]

,count,mean,min,max
region,,,,
east (>= -58°),401.0,1054.081125,642.001820,1493.097093
west (< -58°),481.0,567.435641,478.641302,867.609698


In [9]:
# Seasonal cycle: should be wet in Oct–Mar and dry in Jun–Aug
df.groupby(df["date"].dt.month)["PREC"].mean().round(1).rename("mean PREC by month")

date
1      76.3
2      70.2
3      68.3
4      63.0
5      57.0
6      33.3
7      41.7
8      46.7
9      51.0
10     87.1
11    102.8
12     91.3
Name: mean PREC by month, dtype: float64

## 4. Conclusion

- `PREC` is a **monthly total in mm/month**: the annual sums match observed Paraguayan rainfall, while mm/day would give about 24,000 mm/yr, which is impossible.
- West (Chaco) averages about 570 mm/yr and east about 1,050 mm/yr. The gradient is correct, but the east is drier than observed (a possible model bias; see `docs/data_notes.md`).
- Values are ≥ 0 with no nulls; the maximum is about 470 mm in a single month, which is plausible for a wet-season month.
- To get annual rainfall, **sum** the 12 months (do not average them and multiply by 365).